# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kifayat-Sayed/flyrank-ml-internship1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# ML-09 — Validation and Research Claim Audit

**Goal:** audit the claims in the FlyRank worked-example paper and then audit my own ML-08 model.

**Dataset:** `content_refresh_anonymized.csv` — 30,000 content rows across 32 pseudonymized clients. The target is `is_declining_label = (trend_direction == "down")`.

**Scope:** this is a validation exercise on the starter snapshot. It is not a causal study and does not establish that refreshing a page changes its performance. Results are decision-support evidence only.

[Reference paper in the repository](https://github.com/Kifayat-Sayed/kifayat-flyrank-ml-internship1/blob/main/docs/paper-examples/index.html)

## 1. Two paper findings + my methodology questions

### Finding 1 — the model's advantage depends on queue depth (paper §6.1)

The paper reports that, across five client-grouped folds, HistGradientBoosting had mean Precision@50 of **0.504 ± 0.211**, compared with **0.384 ± 0.182** for the frozen rule. It also reports that the model's observed mean was higher at smaller queue depths but not past approximately K=100, and that it was higher at Precision@50 in only three of five folds.

**Constructive methodology question:** Could the authors show the per-fold label prevalence and the full Precision@K values alongside uncertainty intervals? That would help readers see how much of the measured difference reflects client composition, and whether the proposed queue size is the operating point the evidence best supports.

### Finding 2 — random-row validation overstates generalisation across clients (paper §6.2)

The paper describes a **separate, earlier nine-feature Random Forest experiment**: random-row validation scored 0.996 while client-grouped validation scored 0.496. The paper explicitly distinguishes this boundary experiment from its main eleven-feature HistGradientBoosting result.

**Constructive methodology question:** Could the paper make the exact experiment configuration and both split definitions easy to reproduce from a single linked result artifact? That would help ensure readers do not mistakenly compare those boundary-experiment scores with the main model's cross-validation or sealed-month results.

**Reading note:** These are the paper's reported findings, not results independently reproduced in this notebook. The questions ask for clarity and reproducibility, not for a verdict on the paper.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import warnings
import subprocess
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
REPO_URL = "https://github.com/Kifayat-Sayed/kifayat-flyrank-ml-internship.git"
DATA_CANDIDATES = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("content_refresh_anonymized.csv"),
    Path("/mnt/data/content_refresh_anonymized.csv"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    # In a fresh Colab runtime, clone the public repository to obtain its starter CSV.
    clone_dir = Path("kifayat-flyrank-ml-internship")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)], check=True)
    DATA_PATH = clone_dir / "data/raw/content_refresh_anonymized.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH}")

data = pd.read_csv(DATA_PATH)
required = {"content_id", "client_id", "trend_direction", "trend_pct",
            "days_since_last_update", "impressions_90d"}
missing = required - set(data.columns)
if missing:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")

data["is_declining_label"] = data["trend_direction"].astype(str).str.lower().eq("down").astype(int)
print(f"Loaded: {len(data):,} rows × {data.shape[1]:,} source columns")
print(f"Clients: {data['client_id'].nunique():,}")
print(f"Observed decline base rate: {data['is_declining_label'].mean():.1%}")
print("Label definition: trend_direction == 'down'; trend_direction is itself derived from trend_pct.")

Loaded: 30,000 rows × 45 source columns
Clients: 32
Observed decline base rate: 54.2%
Label definition: trend_direction == 'down'; trend_direction is itself derived from trend_pct.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split (before/after)

I compare two validation designs using the **same Random Forest pipeline, feature list, seed, target definition, and Precision@20 / Precision@50 metrics**:

- **Before — random-row split:** a useful diagnostic for the split-sensitivity demonstration, but rows from the same client can appear in both partitions. It does not establish performance on unseen clients.
- **After — client-grouped split:** all rows from each selected client stay in one partition. This tests ranking on clients not seen during training.

The Week-4 rule is scored on each split as a transparent baseline: `days_since_last_update >= 180` AND `impressions_90d >= 3,000`, ranked by impressions. The dataset is a single snapshot, so a grouped split does **not** solve the separate issue of feature-window overlap with the target.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
TARGET = "is_declining_label"
# Remove IDs, target/siblings, direct ingredients of the label, and near-window activity
# fields to reduce obvious leakage. The source dictionary defines the label from
# impressions_last_30d vs impressions_prev_30d; trend_pct and trend_direction are siblings.
EXCLUDED = {
    "content_id", "client_id", TARGET, "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
    "provider_used", "model_used"
}
feature_cols = [c for c in data.columns if c not in EXCLUDED]
# Remove all-missing features before preprocessing; do not drop rows based on target.
feature_cols = [c for c in feature_cols if not data[c].isna().all()]
X = data[feature_cols].copy()
y = data[TARGET].astype(int)
groups = data["client_id"]

numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

def make_model():
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True))
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10))
    ])
    prep = ColumnTransformer([
        ("numeric", numeric_pipe, numeric_features),
        ("categorical", categorical_pipe, categorical_features)
    ])
    return Pipeline([
        ("preprocess", prep),
        ("classifier", RandomForestClassifier(
            n_estimators=120, min_samples_leaf=5,
            class_weight="balanced_subsample", n_jobs=-1, random_state=SEED
        ))
    ])

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    top = np.argsort(-scores, kind="stable")[:min(k, len(y_true))]
    return float(y_true[top].mean())

def score_frame(name, y_true, model_scores, baseline_scores):
    return [
        {"Split": name, "Method": "Week-4 rule baseline",
         "Precision@20": precision_at_k(y_true, baseline_scores, 20),
         "Precision@50": precision_at_k(y_true, baseline_scores, 50),
         "Test rows": len(y_true), "Base rate": float(np.mean(y_true))},
        {"Split": name, "Method": "Random Forest",
         "Precision@20": precision_at_k(y_true, model_scores, 20),
         "Precision@50": precision_at_k(y_true, model_scores, 50),
         "Test rows": len(y_true), "Base rate": float(np.mean(y_true))}
    ]

# Before: random-row split. Deliberately shown as a diagnostic, not the final estimate.
row_train_idx, row_test_idx = train_test_split(
    np.arange(len(data)), test_size=0.20, random_state=SEED, stratify=y
)
row_model = make_model()
row_model.fit(X.iloc[row_train_idx], y.iloc[row_train_idx])
row_scores = row_model.predict_proba(X.iloc[row_test_idx])[:, 1]
row_test = data.iloc[row_test_idx]
row_base = (
    pd.to_numeric(row_test["days_since_last_update"], errors="coerce").ge(180).fillna(False).astype(int)
    * pd.to_numeric(row_test["impressions_90d"], errors="coerce").fillna(0).ge(3000).astype(int)
    * pd.to_numeric(row_test["impressions_90d"], errors="coerce").fillna(0)
)
results = score_frame("Random-row (diagnostic)", y.iloc[row_test_idx], row_scores, row_base)

# After: grouped client holdout. No client can appear in both partitions.
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
group_train_idx, group_test_idx = next(gss.split(X, y, groups=groups))
train_clients = set(groups.iloc[group_train_idx])
test_clients = set(groups.iloc[group_test_idx])
assert train_clients.isdisjoint(test_clients), "Client overlap detected."

group_model = make_model()
group_model.fit(X.iloc[group_train_idx], y.iloc[group_train_idx])
group_scores = group_model.predict_proba(X.iloc[group_test_idx])[:, 1]
group_test = data.iloc[group_test_idx]
group_y = y.iloc[group_test_idx]
group_base = (
    pd.to_numeric(group_test["days_since_last_update"], errors="coerce").ge(180).fillna(False).astype(int)
    * pd.to_numeric(group_test["impressions_90d"], errors="coerce").fillna(0).ge(3000).astype(int)
    * pd.to_numeric(group_test["impressions_90d"], errors="coerce").fillna(0)
)
results += score_frame("Client-grouped (honest split)", group_y, group_scores, group_base)
comparison = pd.DataFrame(results)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
display(comparison.assign(**{"Precision@20": comparison["Precision@20"].map(lambda v: f"{v:.1%}"), "Precision@50": comparison["Precision@50"].map(lambda v: f"{v:.1%}"), "Base rate": comparison["Base rate"].map(lambda v: f"{v:.1%}")}))
print(f"Random-row test rows: {len(row_test):,}; shared clients between row train/test: {len(set(groups.iloc[row_train_idx]) & set(groups.iloc[row_test_idx])):,}")
print(f"Grouped train: {len(group_train_idx):,} rows / {len(train_clients):,} clients")
print(f"Grouped test:  {len(group_test_idx):,} rows / {len(test_clients):,} clients")
print(f"Grouped client overlap: {len(train_clients & test_clients)}")
print(f"Grouped test decline base rate: {group_y.mean():.1%}")

,Split,Method,Precision@20,Precision@50,Test rows,Base rate
0,Random-row (diagnostic),Week-4 rule baseline,60.0%,64.0%,6000,54.2%
1,Random-row (diagnostic),Random Forest,100.0%,98.0%,6000,54.2%
2,Client-grouped (honest split),Week-4 rule baseline,80.0%,70.0%,6163,51.1%
3,Client-grouped (honest split),Random Forest,65.0%,62.0%,6163,51.1%


Random-row test rows: 6,000; shared clients between row train/test: 31
Grouped train: 23,837 rows / 25 clients
Grouped test:  6,163 rows / 7 clients
Grouped client overlap: 0
Grouped test decline base rate: 51.1%


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

The data dictionary documents a concrete leakage path: `trend_direction` is derived from `trend_pct`, and `trend_pct` is calculated from the most recent and previous 30-day impression totals. The target is `trend_direction == "down"`. Those columns, their direct label ingredients, and IDs are therefore excluded.

A second issue is **time-window alignment**. The source dictionary says the remaining activity metrics are trailing 90-day aggregates ending at export time. A client-grouped split tests transfer to unseen clients, but it does not make these snapshot features temporally prior to the label. Without a historical feature snapshot and a later label window, this notebook cannot certify a deployment-valid time-forward estimate. This limitation is disclosed rather than hidden.

Missing values are imputed within the training pipeline. Numeric missingness indicators are retained; categorical missing values use the training-partition most-frequent value. The dictionary warns that missingness is systematic by content type, so these indicators may carry real data-collection/category information and should be interpreted carefully.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
label_siblings = {"trend_direction", "trend_pct"}
direct_label_inputs = {
    "impressions_last_30d", "impressions_prev_30d"
}
identifiers = {"content_id", "client_id"}
forbidden = {TARGET} | label_siblings | direct_label_inputs | identifiers
leakage_audit = pd.DataFrame([
    {"Check": "Target is constructed from trend_direction == 'down'",
     "Result": "PASS", "Evidence": "Target code shown above; source dictionary documents this definition."},
    {"Check": "Target / sibling fields excluded from predictors",
     "Result": "PASS" if forbidden.isdisjoint(feature_cols) else "FAIL",
     "Evidence": f"Forbidden columns found in feature list: {sorted(forbidden.intersection(feature_cols))}"},
    {"Check": "IDs excluded from predictors",
     "Result": "PASS" if identifiers.isdisjoint(feature_cols) else "FAIL",
     "Evidence": f"ID columns found in feature list: {sorted(identifiers.intersection(feature_cols))}"},
    {"Check": "Direct impression ingredients excluded",
     "Result": "PASS" if direct_label_inputs.isdisjoint(feature_cols) else "FAIL",
     "Evidence": f"Direct label ingredients found in feature list: {sorted(direct_label_inputs.intersection(feature_cols))}"},
    {"Check": "Client-grouped split has zero client overlap",
     "Result": "PASS" if train_clients.isdisjoint(test_clients) else "FAIL",
     "Evidence": f"Train clients={len(train_clients)}; test clients={len(test_clients)}; overlap={len(train_clients & test_clients)}"},
    {"Check": "Feature windows proven strictly earlier than label window",
     "Result": "NOT ESTABLISHED",
     "Evidence": "The supplied file is a single snapshot of trailing-90-day aggregates; no historical decision timestamp is available here."},
])
display(leakage_audit)
print(f"Feature count: {len(feature_cols)}")
print("Excluded columns present in source:", sorted(EXCLUDED.intersection(data.columns)))
print("Numeric features:", len(numeric_features), "| categorical features:", len(categorical_features))

,Check,Result,Evidence
0,Target is constructed from trend_direction == ...,PASS,Target code shown above; source dictionary doc...
1,Target / sibling fields excluded from predictors,PASS,Forbidden columns found in feature list: []
2,IDs excluded from predictors,PASS,ID columns found in feature list: []
3,Direct impression ingredients excluded,PASS,Direct label ingredients found in feature list...
4,Client-grouped split has zero client overlap,PASS,Train clients=25; test clients=7; overlap=0
5,Feature windows proven strictly earlier than l...,NOT ESTABLISHED,The supplied file is a single snapshot of trai...


Feature count: 32
Excluded columns present in source: ['clicks_last_30d', 'clicks_prev_30d', 'client_id', 'content_id', 'impressions_last_30d', 'impressions_prev_30d', 'is_declining_label', 'model_used', 'provider_used', 'sessions_last_30d', 'sessions_prev_30d', 'trend_direction', 'trend_pct']
Numeric features: 23 | categorical features: 9


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

### Claim that goes too far

> “The Random Forest is better than the baseline and predicts which pages will decline.”

This overstates the evidence because the result depends on the split, the evaluation uses a single snapshot, and the available feature windows are not proven to precede the target window.

### Safer public-facing wording

> “In this 30,000-row anonymized snapshot, the Random Forest's measured Precision@20 and Precision@50 were compared with the Week-4 rule using both a random-row diagnostic split and a client-grouped holdout. The grouped results are the more relevant check for ranking pages from unseen clients. These results are directional decision-support evidence, not proof of future performance or evidence that refreshing a flagged page will improve outcomes.”

The numeric comparison below is generated from this run. Do not copy a score into a public summary without stating the split and base rate.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Grouped-holdout error examples at the fixed 0.50 probability threshold.
group_pred = (group_scores >= 0.50).astype(int)
error_rows = group_test[["content_id", "trend_direction",
                         "days_since_last_update", "impressions_90d"]].copy()
error_rows["actual_declining"] = group_y.to_numpy()
error_rows["model_score"] = group_scores
error_rows["predicted_declining_at_0_50"] = group_pred
error_rows["error_type"] = np.select(
    [(error_rows["actual_declining"].eq(0) & error_rows["predicted_declining_at_0_50"].eq(1)),
     (error_rows["actual_declining"].eq(1) & error_rows["predicted_declining_at_0_50"].eq(0))],
    ["false positive", "false negative"],
    default="correct"
)
errors = error_rows[error_rows["error_type"] != "correct"].copy()
print("Confusion matrix at threshold 0.50 [rows=true 0/1, columns=predicted 0/1]:")
print(confusion_matrix(group_y, group_pred, labels=[0, 1]))
print("\nClassification report:")
print(classification_report(group_y, group_pred, zero_division=0))
print(f"Misclassified grouped-test rows: {len(errors):,} / {len(group_test):,}")

print("\nConcrete false-positive examples (predicted decline; observed label is not down):")
display(errors[errors["error_type"] == "false positive"].sort_values("model_score", ascending=False)[["content_id", "trend_direction", "days_since_last_update", "impressions_90d", "actual_declining", "model_score", "error_type"]].head(3))
print("Concrete false-negative examples (missed observed decline):")
display(errors[errors["error_type"] == "false negative"].sort_values("model_score")[["content_id", "trend_direction", "days_since_last_update", "impressions_90d", "actual_declining", "model_score", "error_type"]].head(3))

# Summarize the measured random-vs-grouped gap for the model, without claiming causation.
metric_summary = comparison[comparison["Method"] == "Random Forest"].set_index("Split")
for metric in ["Precision@20", "Precision@50"]:
    random_value = metric_summary.loc["Random-row (diagnostic)", metric]
    grouped_value = metric_summary.loc["Client-grouped (honest split)", metric]
    print(f"{metric}: random-row={random_value:.1%}; grouped={grouped_value:.1%}; "
          f"grouped minus random={100*(grouped_value-random_value):+.1f} percentage points")

Confusion matrix at threshold 0.50 [rows=true 0/1, columns=predicted 0/1]:
[[1724 1290]
 [1239 1910]]

Classification report:
              precision    recall  f1-score   support

           0       0.58      0.57      0.58      3014
           1       0.60      0.61      0.60      3149

    accuracy                           0.59      6163
   macro avg       0.59      0.59      0.59      6163
weighted avg       0.59      0.59      0.59      6163

Misclassified grouped-test rows: 2,529 / 6,163

Concrete false-positive examples (predicted decline; observed label is not down):


,content_id,trend_direction,days_since_last_update,impressions_90d,actual_declining,model_score,error_type
22042,content_2ba626fea4d6,up,104,360,0,0.941754,false positive
22526,content_1d0963b56227,up,104,3445,0,0.924522,false positive
4050,content_500bd3907331,stable,104,4037,0,0.914394,false positive


Concrete false-negative examples (missed observed decline):


,content_id,trend_direction,days_since_last_update,impressions_90d,actual_declining,model_score,error_type
1864,content_16f38acf0f26,down,20,2,1,0.070787,false negative
13114,content_8f222654e93f,down,20,2,1,0.084075,false negative
25350,content_a4c38287770e,down,20,2,1,0.085139,false negative


Precision@20: random-row=100.0%; grouped=65.0%; grouped minus random=-35.0 percentage points
Precision@50: random-row=98.0%; grouped=62.0%; grouped minus random=-36.0 percentage points


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-check

- Two paper findings and constructive methodology questions are documented.
- Random-row and client-grouped results are measured with the same model and ranking metrics.
- Group integrity is asserted in code.
- Label-derived fields, direct label ingredients, and identifiers are audited.
- Real grouped-test false-positive and false-negative examples are displayed.
- The strongest claim is rewritten using observed/measured/directional/decision-support language.
- The single-snapshot temporal limitation is explicitly disclosed.
- After running in your repository/Colab environment, verify every cell succeeds and then commit this file as `work/notebooks/w06_validation_audit.ipynb`.

**Important limitation:** the grouped split improves the client boundary, but a future-facing evaluation needs historical features that end before a later label window. This dataset alone cannot establish that temporal boundary.